<a href="https://colab.research.google.com/github/Arohi-jd/aiml/blob/main/Question_CodingAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coding Agent

In this exercise, you will build a **Copilot-style coding agent** using LangChain and LangGraph that:
1. Generates Python code from a plain-English description
2. Saves the output to disk — as a single script or a structured multi-file project
3. Runs the code in a safe sandbox subprocess
4. Automatically fixes bugs and retries if the code fails
5. Optionally writes pytest unit tests

### What is a ReAct Agent?

**ReAct = Reasoning + Acting.** Unlike a fixed pipeline, a ReAct agent follows a dynamic loop:

```
User Prompt → Agent (LLM reasons) → Tool Call → Tool Result (observation) → Agent (reasons again) → ... → Final Answer
```

The **LLM decides** which tool to call and when to stop. This is what makes it an *agent* rather than a pipeline.

### Key LangGraph Concepts You Will Implement:

| Concept | Description |
|---|---|
| **Tools** (`@tool`) | Functions the LLM can choose to call (generate, fix, run, save code) |
| **Tool Binding** (`bind_tools`) | Gives the LLM awareness of available tools |
| **CodingState** (`TypedDict + add_messages`) | Shared state that accumulates message history across the loop |
| **Agent Node** | The LLM reasoning step — reads state, decides to call a tool or give a final answer |
| **Tool Node** (`ToolNode`) | Automatically executes whichever tool the LLM chose |
| **Conditional Edges** | Routes the graph based on whether the LLM made a tool call or not |
| **Message Trimming** | Keeps context short by only passing recent messages to the LLM |

### Single-File vs. Multi-File Decision:

The agent autonomously decides the output structure based on your request:

```
Simple script      →  output/script.py
Multi-module project  →  output/project/main.py
                          output/project/utils.py
```

## Step 1: Install & Import Dependencies

Install all required packages and import the necessary modules.

**Libraries used:**
- `langgraph`, `langchain`, `langchain-groq` — LLM agent framework
- `langchain-core` — tools, messages, and prompts

In [ ]:
%pip install -q langgraph langchain-groq langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.1 MB/s eta 0:00:00


In [ ]:
import os
import getpass
import tempfile
import subprocess
from typing import TypedDict, Annotated
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from IPython.display import Image, display, Markdown

print("Imports done.")

## Step 2: Initialize the LLM

Initialize `ChatGroq` with your Groq API key and verify the connection.

**Hints:**
- `ChatGroq(model=..., temperature=0)` — temperature 0 gives deterministic tool-call decisions
- Invoke the LLM with a `HumanMessage` and print `response.content` to verify

**Documentation:**
- [ChatGroq](https://python.langchain.com/docs/integrations/chat/groq/)

In [ ]:
# Set your Groq API key
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
# Initialize the LLM
llm = ______(model="llama-3.3-70b-versatile", temperature=______ , api_key=_______)

# Quick connection test
response = llm.invoke([HumanMessage(content="Say 'LLM ready!' and nothing else.")])
print(response.______)

## Step 3: Define the Copilot Tools

In a ReAct agent, **tools** are the actions the LLM can choose to take. Each `@tool`-decorated function becomes available to the agent. The **docstring** of each tool is what the LLM reads to understand when to use it.

You will define 7 tools:

| Tool | Purpose |
|---|---|
| `generate_code` | Write Python code for a single file from a description |
| `explain_code` | Explain what a piece of code does |
| `fix_bug` | Find and fix bugs, return corrected code |
| `write_tests` | Generate pytest unit tests |
| `run_code` | Run code inline in a temp-file sandbox |
| `save_output` | Save code to `output/` directory (single file or project folder) |
| `run_file` | Run a saved file from its own directory (fixes imports for multi-file projects) |

**Hints:**
- Use the `@tool` decorator from `langchain_core.tools`
- The `llm.invoke([SystemMessage(...), HumanMessage(...)])` pattern passes context + request to the LLM
- For `run_code`: use `tempfile.NamedTemporaryFile` to write code to disk, run with `subprocess.run(["python3", tmp_path], ...)`, always delete the temp file in a `finally` block
- For `save_output`: `os.path.join("output", path)` builds the full path; `os.makedirs(..., exist_ok=True)` creates intermediate folders
- For `run_file`: use `cwd=work_dir` in `subprocess.run(...)` so that sibling module imports resolve correctly

**Documentation:**
- [LangChain @tool decorator](https://python.langchain.com/docs/how_to/custom_tools/#tool-decorator)
- [subprocess.run](https://docs.python.org/3/library/subprocess.html#subprocess.run)
- [tempfile.NamedTemporaryFile](https://docs.python.org/3/library/tempfile.html#tempfile.NamedTemporaryFile)

In [ ]:
# Tool 1 — Generate code for a single file from a plain-English description
@______
def generate_code(description: str) -> str:
    """Write Python code for a single file based on a plain-English description.
    For multi-file projects, call this tool once per file with a description specific to that module.
    """
    response = llm.invoke([
        SystemMessage(content=(
            "You are an expert Python developer. Write code for ONE file only. "
            "Return only clean Python code with no explanation and no markdown fences. "
            "For modules that will be imported, do not add an if __name__ == '__main__' block."
        )),
        HumanMessage(content=f"Write Python code for: {______}")
    ])
    return response.content

In [ ]:
# Tool 2 — Explain what a piece of code does
@______
def explain_code(code: str) -> str:
    """Explain what a given piece of Python code does in simple terms."""
    response = llm.invoke([
        SystemMessage(content="You are a coding teacher. Explain code clearly in 3-5 sentences for a beginner."),
        HumanMessage(content=f"Explain this code:\n\n{______}")
    ])
    return response.______


# Tool 3 — Find and fix bugs in code
@______
def fix_bug(code: str) -> str:
    """Find and fix bugs in the given Python code. Returns the corrected code with a brief note on what was fixed."""
    response = llm.invoke([
        SystemMessage(content="You are a debugging expert. Identify the bug, fix it, and return the corrected code followed by a one-line note: '# Fix: ...'"),
        HumanMessage(content=f"Find and fix the bug in this code:\n\n{code}")
    ])
    return response.content


# Tool 4 — Write pytest unit tests for code
@______
def write_tests(code: str) -> str:
    """Generate pytest unit tests for the given Python code."""
    response = llm.invoke([
        SystemMessage(content="You are a QA engineer. Write clear pytest unit tests. Return only the test code."),
        HumanMessage(content=f"Write pytest tests for:\n\n{code}")
    ])
    return response.content

In [ ]:
# Tool 5 — Run Python code inline in a sandbox subprocess
@______
def run_code(code: str) -> str:
    """Execute Python code in a sandboxed subprocess and return the output or error."""
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(______)
        tmp = f.name
    try:
        result = subprocess.run(["python3", ______], capture_output=True, text=True, timeout=10)
        return result.stdout if result.returncode == 0 else f"Error:\n{result.______}"
    except subprocess.TimeoutExpired:
        return "Error: Code timed out after 10 seconds."
    finally:
        os.unlink(______)  # Always delete the temp file


# Tool 6 — Save code to disk (single file or project folder)
@______
def save_output(path: str, code: str) -> str:
    """Save code to a file on disk under an 'output/' directory.

    Rules for the 'path' argument:
      - Simple script  → plain filename,     e.g. 'sorter.py'
      - Project module → subfolder path,     e.g. 'calculator/main.py'

    For a multi-file project call this tool once per file.
    Returns the full path where the file was saved.
    """
    full_path = os.path.join("______", path)
    os.makedirs(os.path.dirname(______), exist_ok=True)
    with open(full_path, "w") as f:
        f.write(code)
    return f"Saved: {______}"


# Tool 7 — Run a saved file from its own directory (fixes imports for multi-file projects)
@______
def run_file(path: str) -> str:
    """Run a saved Python file from the output/ directory.

    Use this instead of run_code when the task involves multiple files,
    because it sets the working directory to the file's folder so that
    local imports resolve correctly.

    path: relative path inside output/, e.g. 'calculator/main.py'
    """
    full_path = os.path.abspath(os.path.join("output", path))
    work_dir = os.path.________(full_path)  # directory containing the file
    if not os.path.exists(full_path):
        return f"Error: File not found at {full_path}. Make sure save_output was called first."
    try:
        result = subprocess.run(
            ["python3", full_path],
            capture_output=True, text=True, timeout=10, cwd=______
        )
        return result.stdout if result.returncode == 0 else f"Error:\n{result.stderr}"
    except subprocess.TimeoutExpired:
        return "Error: Code timed out after 10 seconds."

print("All tools defined.")

## Step 4: Define the Agent State and Bind Tools

A LangGraph agent communicates through **state** — a typed dictionary that is passed between every node.

- `messages` accumulates the full conversation via `add_messages` (an append-only reducer)
- `steps` is a counter used to prevent the agent from looping forever

Once `CodingState` is defined, use `llm.bind_tools(tools)` to create an LLM instance that knows about all 7 tools.

**Hints:**
- `TypedDict` defines the shape of state; import it from `typing`
- `Annotated[list, add_messages]` is the type annotation that LangGraph uses to know it should **append** new messages instead of overwriting them
- `llm.bind_tools(tools)` returns a new LLM instance with tool schemas attached; assign it to `llm_with_tools`

**Documentation:**
- [LangGraph state schema](https://langchain-ai.github.io/langgraph/concepts/low_level/#state)
- [add_messages reducer](https://langchain-ai.github.io/langgraph/reference/graphs/#langgraph.graph.message.add_messages)

In [ ]:
# Define the shared state for the agent graph
class CodingState(TypedDict):
    messages: Annotated[______, add_messages]  # append-only conversation history
    steps: ______                              # loop counter

# Collect all tools in one list
tools = [generate_code, explain_code, fix_bug, write_tests, run_code, save_output, run_file]

# Bind tools to the LLM so it can call them
llm_with_tools = llm.______(tools)

print(f"Bound {len(tools)} tools to LLM.")

## Step 5: Define the Agent Node, Tool Node, and Routing Logic

Three pieces connect the ReAct loop:

1. **`trim_messages`** — keeps context short by removing old tool exchanges (keeps the system message + original task + the last N messages)
2. **`agent_node`** — invokes `llm_with_tools` on trimmed state messages; returns `{"messages": [response], "steps": state["steps"] + 1}`
3. **`tool_node`** — a built-in LangGraph node (`ToolNode`) that executes whichever tool the LLM chose
4. **`should_continue`** — a routing function: returns `"tools"` if the LLM called a tool **and** steps < 20, otherwise returns `"end"`

**Hints:**
- `state["messages"][-1]` gives the last message; check `.tool_calls` to see if the LLM requested a tool
- `ToolNode(tools)` handles the JSON-parsing and tool execution automatically
- The step limit (20) prevents infinite loops on hard tasks

**Documentation:**
- [LangGraph ToolNode](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.tool_node.ToolNode)
- [Conditional edges](https://langchain-ai.github.io/langgraph/concepts/low_level/#conditional-edges)

In [ ]:
def trim_messages(messages: list, keep_last: int = 8) -> list:
    """Keep the system message + first user message + the last N messages to avoid context overflow."""
    if len(messages) <= keep_last + 2:
        return ______
    return messages[:2] + messages[-______:]  # system + task + recent history


def agent_node(state: CodingState) -> dict:
    """The LLM reasoning step — reads state, decides to call a tool or give a final answer."""
    trimmed = trim_messages(state["______"])
    response = ______.invoke(trimmed)       # call the tool-aware LLM
    return {
        "messages": [______],               # append the new response
        "steps": state["steps"] + ______    # increment step counter
    }


# ToolNode automatically handles tool execution based on the LLM's tool call
tool_node = ______(tools)


def should_continue(state: CodingState) -> str:
    """Route: call tools if the LLM made a tool call and we haven't hit the step limit."""
    last_message = state["messages"][______]
    if last_message.tool_calls and state["______"] < 20:
        return "______"
    return "______"

## Step 6: Build and Compile the LangGraph Workflow

Now wire the nodes together into a directed graph:

```
START → agent → (should_continue) → tools → agent   (loop)
                               ↓
                              END
```

**Hints:**
- `StateGraph(CodingState)` creates a graph that uses your state schema
- `.add_node("name", function)` registers a node
- `.add_edge(START, "agent")` draws a fixed edge from START to agent
- `.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})` draws a branching edge
- `.add_edge("tools", "agent")` loops tool results back to the agent
- `.compile()` finalizes the graph — save it to `workflow`
- `workflow.get_graph().draw_mermaid_png()` renders a diagram

**Documentation:**
- [StateGraph API](https://langchain-ai.github.io/langgraph/reference/graphs/#langgraph.graph.state.StateGraph)

In [ ]:
# Build the graph
graph_builder = ______(CodingState)

# Register nodes
graph_builder.add_node("______", agent_node)
graph_builder.add_node("______", tool_node)

# Wire edges
graph_builder.add_edge(______, "agent")
graph_builder.add_conditional_edges(
    "agent",
    ______,                                         # routing function
    {"tools": "______", "end": ______}              # map return values to node names
)
graph_builder.add_edge("tools", "______")           # loop back to agent

# Compile
workflow = graph_builder.______()

# Visualize the graph
display(Image(workflow.get_graph().draw_mermaid_png()))
print("Graph compiled.")

## Step 7: Create the System Prompt and Runner Function

The **system prompt** is the high-level instruction that tells the agent how to behave — it describes both the SINGLE-FILE and MULTI-FILE workflows.

The **`run_workflow`** helper function:
1. Wraps the user task in a `HumanMessage`
2. Packages it with the `SystemMessage` into initial state
3. Invokes the compiled workflow
4. Extracts and displays the final answer

**Hints:**
- `workflow.invoke({"messages": [...], "steps": 0})` starts the loop from step 0
- The final answer is in `result["messages"][-1].content`
- Use `display(Markdown(...))` for nicely formatted output

**Documentation:**
- [LangGraph invoke](https://langchain-ai.github.io/langgraph/reference/graphs/#langgraph.graph.state.CompiledStateGraph.invoke)

In [ ]:
SYSTEM_PROMPT = """You are a Copilot-style coding assistant. Your goal is to complete the user's coding task step by step using the available tools.

SINGLE-FILE workflow (simple script or one module):
  1. generate_code  → create the code
  2. save_output    → save it as output/<filename>.py
  3. run_file       → run the saved file and check output
  4. If there is an error, call fix_bug then save_output again, then run_file again
  5. Optionally call write_tests to generate unit tests

MULTI-FILE project workflow (app with multiple modules):
  1. Plan the project structure (e.g. main.py + utils.py)
  2. For each file, call generate_code then save_output (one call per file)
  3. After all files are saved, call run_file on the main entry point (e.g. project/main.py)
  4. If there is an error, call fix_bug then save_output for the affected file, then run_file again

RULES:
- Always save files before running them with run_file
- Use run_code only for quick throwaway checks, not for multi-file projects
- Only call one tool at a time — do not chain tools in a single response
- Stop after the task is confirmed working; do not add unrequested features
- Keep responses concise — no long explanations unless asked
"""

def run_workflow(task: str):
    """Run the coding agent on a plain-English task and display the result."""
    print(f"Task: {task}\n")
    result = ______.invoke({
        "______": [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=______)],
        "steps": ______
    })
    answer = result["messages"][______].content
    display(Markdown(f"**Result:**\n\n{______}"))
    return answer

## Step 8: Run the Agent — Demos

Run the agent on two provided examples. You do not need to fill any blanks here — just execute the cells and observe how the agent behaves.

**Watch for:**
- Which tools does the agent call in what order?
- How many ReAct cycles does it take?
- What files are created in the `output/` directory?

In [ ]:
# Demo 1 — Single-file script: prime checker
run_workflow("Write a Python script that checks if a number is prime and tests it for the numbers 2, 3, 4, 17, and 100.")

In [ ]:
# Demo 2 — Multi-file project: calculator with a utilities module
run_workflow(
    "Create a small calculator project with two files: "
    "calculator/utils.py that provides add, subtract, multiply, divide functions, "
    "and calculator/main.py that imports utils and demonstrates all four operations."
)

## Step 9: Interactive Mode

Run your agent interactively. Type any coding task at the prompt and press Enter. Type `quit` to exit.

Fill in the blanks to wire the `input()` loop to `run_workflow`.

In [ ]:
print("Coding Agent ready. Type a task or 'quit' to exit.\n")
while True:
    task = ______("> ").strip()
    if not task or task.lower() == "______":
        print("Goodbye.")
        break
    ______(task)